# Stage 4 — Group and Cross-Unit Comparison

1. 跨校橋接分析（組層級與全體層級專屬）：跨校邊佔比、E-I index、school assortativity、
   跨校 broker 教授清單、Louvain 社群 vs `school_id` 的 NMI / ARI。
2. 組間與系所間比較：統一比較表（含 coverage）、三組研究領域重疊。
3. `analysis/network-analysis/comparison/figures/` 的 12 張跨單位圖表。
4. 產生 `docs/network-analysis-report.md`。

In [1]:
import json
import math
from collections import Counter
from datetime import datetime, timedelta, timezone
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib_venn import venn3
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from scipy.spatial.distance import squareform
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

ROOT = Path.cwd()
while not (ROOT / "data" / "data-preprocessing").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

NET_DIR = ROOT / "data" / "network-analysis"
OUT_ROOT = ROOT / "analysis" / "network-analysis"
CMP_DIR = OUT_ROOT / "comparison"
CMP_FIG = CMP_DIR / "figures"
CMP_FIG.mkdir(parents=True, exist_ok=True)
DOCS = ROOT / "docs"

GROUPS = {
    "fin-and-econ": ["nccu-fin", "nccu-mb", "nthu-qf", "ntu-econ", "ntu-fin", "nycu-info-mag-fin"],
    "ds": ["ncku-stats-ds", "ntu-comm-engin-ds-network", "ntu-ds"],
    "cs": ["ncku-cs-ie", "nthu-cs", "ntu-cs-ie", "ntu-im", "nycu-cs"],
}
SCHOOL_TO_GROUP = {s: g for g, ss in GROUPS.items() for s in ss}
SCHOOLS = sorted(SCHOOL_TO_GROUP)
LAMBDA = 0.4
SEED = 42

TAXONOMY = pd.read_csv(NET_DIR / "research_taxonomy.csv", encoding="utf-8-sig")
KW2L1 = dict(zip(TAXONOMY["keyword_en"], TAXONOMY["parent_l1"]))
KW2L2 = dict(zip(TAXONOMY["keyword_en"], TAXONOMY["parent_l2"]))
with (NET_DIR / "units_manifest.json").open(encoding="utf-8") as fh:
    MANIFEST = json.load(fh)
META = {m["unit"]: m for m in MANIFEST}

METRICS = {}
for m in MANIFEST:
    with (OUT_ROOT / m["unit"] / "network_metrics.json").open(encoding="utf-8") as fh:
        METRICS[m["unit"]] = json.load(fh)

PALETTE = [
    "#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3", "#937860",
    "#DA8BC3", "#8C8C8C", "#CCB974", "#64B5CD", "#1F77B4", "#FF7F0E",
    "#2CA02C", "#D62728", "#9467BD", "#8C564B", "#E377C2", "#7F7F7F",
    "#BCBD22", "#17BECF",
]
L1_ORDER = sorted(TAXONOMY["parent_l1"].unique())
L1_COLOR = {l1: PALETTE[i % len(PALETTE)] for i, l1 in enumerate(L1_ORDER)}
SCHOOL_COLOR = {s: PALETTE[i % len(PALETTE)] for i, s in enumerate(SCHOOLS)}
GROUP_COLOR = {"cs": "#4C72B0", "ds": "#55A868", "fin-and-econ": "#DD8452"}
COLOR_PROF = "#4C72B0"
COLOR_KW = "#DD8452"

import matplotlib.font_manager as fm
import re

CJK_RE = re.compile(r"[㐀-䶿一-鿿豈-﫿]")
_avail = {f.name for f in fm.fontManager.ttflist}
CJK_FONT = next((c for c in ("Microsoft JhengHei", "Microsoft YaHei", "MingLiU", "SimSun",
                 "PingFang TC", "Heiti TC", "Hiragino Sans",
                 "Noto Sans CJK TC", "Noto Sans TC", "Source Han Sans TW",
                 "WenQuanYi Zen Hei", "Droid Sans Fallback") if c in _avail), None)
FONT_OK = CJK_FONT is not None
plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ([CJK_FONT] if FONT_OK else []) + ["DejaVu Sans", "Arial", "sans-serif"],
        "axes.unicode_minus": False,
        "savefig.dpi": 180,
        "axes.titlesize": 12,
        "axes.labelsize": 10,
        "legend.fontsize": 8,
    }
)
print(f"CJK font: {CJK_FONT}")

LANG_CHECK_ROWS: list[dict] = []
FIGURE_INDEX: list[dict] = []
ALLOWED_CJK: set[str] = set()
for m in MANIFEST:
    nodes = pd.read_csv(ROOT / m["out_dir"] / "professor_nodes.csv", encoding="utf-8-sig")
    ALLOWED_CJK |= {str(v).strip() for v in nodes["professor_name"].dropna()}
    ALLOWED_CJK |= {str(v).strip() for v in nodes["name_en"].dropna()}


def save_figure(fig, path: Path, caption: str | None = None) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if caption:
        fig.text(0.005, 0.005, caption, ha="left", va="bottom", fontsize=8, color="#444444")
    texts: list[str] = []
    for ax in fig.get_axes():
        texts += [ax.get_title(), ax.get_xlabel(), ax.get_ylabel()]
        texts += [t.get_text() for t in ax.get_xticklabels()]
        texts += [t.get_text() for t in ax.get_yticklabels()]
        texts += [t.get_text() for t in ax.texts]
        leg = ax.get_legend()
        if leg is not None:
            texts += [t.get_text() for t in leg.get_texts()]
    texts += [t.get_text() for t in fig.texts]
    for leg in getattr(fig, "legends", []):
        texts += [t.get_text() for t in leg.get_texts()]
    violations = [t.strip() for t in texts if t and CJK_RE.search(t) and t.strip() not in ALLOWED_CJK]
    LANG_CHECK_ROWS.append(
        {
            "unit": "comparison",
            "figure": path.name,
            "n_text_elements": len([t for t in texts if t]),
            "n_cjk_elements": len([t for t in texts if t and CJK_RE.search(t)]),
            "n_violations": len(violations),
            "violations": " | ".join(sorted(set(violations))[:5]),
        }
    )
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    FIGURE_INDEX.append({"unit": "comparison", "figure": path.name,
                         "path": str(path.relative_to(ROOT)).replace("\\", "/")})

CJK font: Microsoft JhengHei


## 4.1 跨校橋接分析（組層級與全體層級）

- 一條 professor–professor 邊若兩端**沒有共同的 `school_id`** 即為跨校邊。
- **E-I index** = (跨校邊 − 校內邊) / (跨校邊 + 校內邊)，範圍 −1（完全校內）到 +1（完全跨校）。
- **Louvain 社群 vs `school_id` 分割的 NMI / ARI** 直接回答「社群結構由學校決定，還是由研究主題決定」。

In [2]:
def load_professor_graph(unit: str) -> nx.Graph:
    return nx.read_graphml(OUT_ROOT / unit / "professor_network.graphml")


def schools_of(G: nx.Graph, n: str) -> set[str]:
    return {s for s in str(G.nodes[n].get("school_ids", "")).split(" | ") if s}


bridging_rows = []
broker_frames = {}
for unit in list(GROUPS) + ["overall"]:
    G = load_professor_graph(unit)
    sch = {n: schools_of(G, n) for n in G.nodes()}
    internal = external = 0
    w_internal = w_external = 0.0
    cross_deg = Counter()
    for u, v, d in G.edges(data=True):
        cross = not (sch[u] & sch[v])
        if cross:
            external += 1
            w_external += d.get("weight", 1.0)
            cross_deg[u] += 1
            cross_deg[v] += 1
        else:
            internal += 1
            w_internal += d.get("weight", 1.0)
    total = internal + external
    ei = (external - internal) / total if total else float("nan")
    ei_w = (w_external - w_internal) / (w_external + w_internal) if (w_external + w_internal) else float("nan")

    for n in G.nodes():
        G.nodes[n]["school_primary"] = sorted(sch[n])[0] if sch[n] else "unknown"
    try:
        assort_school = float(nx.attribute_assortativity_coefficient(G, "school_primary"))
    except Exception:
        assort_school = float("nan")

    comm = {n: int(G.nodes[n].get("community", -1)) for n in G.nodes()}
    labels = {n: G.nodes[n]["school_primary"] for n in G.nodes()}
    keys = sorted(comm)
    nmi = float(normalized_mutual_info_score([labels[k] for k in keys], [comm[k] for k in keys]))
    ari = float(adjusted_rand_score([labels[k] for k in keys], [comm[k] for k in keys]))

    l1 = {n: G.nodes[n].get("dominant_l1", "") for n in G.nodes()}
    nmi_l1 = float(normalized_mutual_info_score([l1[k] for k in keys], [comm[k] for k in keys]))
    ari_l1 = float(adjusted_rand_score([l1[k] for k in keys], [comm[k] for k in keys]))

    btw = nx.betweenness_centrality(G, weight="distance", normalized=True, seed=SEED)
    deg = dict(G.degree())
    joint = {n for n in G.nodes() if int(G.nodes[n].get("n_appointments", 1)) > 1}
    brokers = pd.DataFrame(
        [
            {
                "unit": unit,
                "node_id": n,
                "professor_name": G.nodes[n].get("professor_name", ""),
                "name_en": G.nodes[n].get("name_en", ""),
                "school_ids": G.nodes[n].get("school_ids", ""),
                "n_appointments": int(G.nodes[n].get("n_appointments", 1)),
                "joint_appointment": n in joint,
                "degree": deg[n],
                "n_cross_school_edges": cross_deg.get(n, 0),
                "cross_school_edge_share": cross_deg.get(n, 0) / deg[n] if deg[n] else 0.0,
                "betweenness": btw[n],
                "dominant_l1": G.nodes[n].get("dominant_l1", ""),
            }
            for n in G.nodes()
            if cross_deg.get(n, 0) > 0
        ]
    ).sort_values("betweenness", ascending=False).reset_index(drop=True)
    broker_frames[unit] = brokers

    out_dir = OUT_ROOT / unit
    brokers.to_csv(out_dir / "cross_school_bridging.csv", index=False, encoding="utf-8-sig")

    bridging_rows.append(
        {
            "unit": unit,
            "level": META[unit]["level"],
            "n_schools": len(META[unit]["school_ids"]),
            "n_professors": G.number_of_nodes(),
            "coverage_pct": META[unit]["coverage_pct"],
            "n_edges": total,
            "n_internal_edges": internal,
            "n_cross_school_edges": external,
            "cross_school_edge_share": external / total if total else float("nan"),
            "cross_school_weight_share": w_external / (w_external + w_internal) if total else float("nan"),
            "ei_index": ei,
            "ei_index_weighted": ei_w,
            "school_assortativity": assort_school,
            "community_vs_school_nmi": nmi,
            "community_vs_school_ari": ari,
            "community_vs_dominant_l1_nmi": nmi_l1,
            "community_vs_dominant_l1_ari": ari_l1,
            "n_joint_appointment_professors": len(joint),
            "n_cross_school_brokers": len(brokers),
        }
    )

bridging_df = pd.DataFrame(bridging_rows)
bridging_df.to_csv(CMP_DIR / "cross_school_bridging_summary.csv", index=False, encoding="utf-8-sig")
print(bridging_df.round(4).to_string(index=False))

        unit   level  n_schools  n_professors  coverage_pct  n_edges  n_internal_edges  n_cross_school_edges  cross_school_edge_share  cross_school_weight_share  ei_index  ei_index_weighted  school_assortativity  community_vs_school_nmi  community_vs_school_ari  community_vs_dominant_l1_nmi  community_vs_dominant_l1_ari  n_joint_appointment_professors  n_cross_school_brokers
fin-and-econ   group          6           148          83.0     3720              1019                  2701                   0.7261                     0.7046    0.4522             0.4093                0.1069                   0.3046                   0.2516                        0.4772                        0.3310                               2                     147
          ds   group          3           107          87.0     1555               975                   580                   0.3730                     0.3444   -0.2540            -0.3113                0.2429                   0.3112        

### 合聘教授（`n_appointments > 1`）

合聘教授必然是跨校節點，必須與「研究主題造成的跨校連結」分開看。

In [3]:
joint_rows = []
for unit in list(GROUPS) + ["overall"]:
    b = broker_frames[unit]
    j = b[b["joint_appointment"]]
    joint_rows.append(
        {
            "unit": unit,
            "n_joint": len(j),
            "n_brokers": len(b),
            "joint_share_of_brokers": len(j) / len(b) if len(b) else float("nan"),
            "cross_edges_from_joint": int(j["n_cross_school_edges"].sum()),
            "cross_edges_total": int(b["n_cross_school_edges"].sum() / 2),
            "mean_betweenness_joint": float(j["betweenness"].mean()) if len(j) else float("nan"),
            "mean_betweenness_nonjoint": float(b[~b["joint_appointment"]]["betweenness"].mean()),
        }
    )
joint_df = pd.DataFrame(joint_rows)
joint_df.to_csv(CMP_DIR / "joint_appointment_summary.csv", index=False, encoding="utf-8-sig")
print(joint_df.round(4).to_string(index=False))

        unit  n_joint  n_brokers  joint_share_of_brokers  cross_edges_from_joint  cross_edges_total  mean_betweenness_joint  mean_betweenness_nonjoint
fin-and-econ        2        147                  0.0136                      88               2701                  0.0252                     0.0084
          ds        9        102                  0.0882                      78                580                  0.0371                     0.0143
          cs        5        379                  0.0132                     286              13881                  0.0042                     0.0037
     overall       37        616                  0.0601                    4378              24967                  0.0085                     0.0027


## 4.2 組間與系所間比較

統一比較表：density、mean degree、LCC 佔比、modularity Q、社群數、平均 clustering、
領域集中度（entropy 與 Gini）、跨校邊比例、**coverage**。

In [4]:
def entropy_and_gini(counts: np.ndarray) -> tuple[float, float]:
    counts = np.asarray(counts, dtype=float)
    counts = counts[counts > 0]
    if len(counts) == 0:
        return float("nan"), float("nan")
    p = counts / counts.sum()
    ent = float(-(p * np.log(p)).sum())
    ent_norm = ent / math.log(len(counts)) if len(counts) > 1 else 0.0
    x = np.sort(counts)
    n = len(x)
    gini = float((2 * np.arange(1, n + 1) - n - 1).dot(x) / (n * x.sum()))
    return ent_norm, gini


def unit_row(unit: str) -> dict:
    m = METRICS[unit]
    kwn = pd.read_csv(ROOT / META[unit]["out_dir"] / "keyword_nodes.csv", encoding="utf-8-sig")
    ent, gini = entropy_and_gini(kwn["n_faculty_in_unit"].values)
    bridge = bridging_df[bridging_df["unit"] == unit]
    return {
        "unit": unit,
        "level": m["level"],
        "group": SCHOOL_TO_GROUP.get(unit, unit),
        "n_professors": m["bipartite"]["n_professors"],
        "n_excluded_no_keywords": m["coverage"]["n_excluded_no_keywords"],
        "coverage_pct": m["coverage"]["coverage_pct"],
        "n_keywords": m["bipartite"]["n_keywords"],
        "bipartite_density": m["bipartite"]["density"],
        "lcc_node_share": m["bipartite"]["lcc_node_share"],
        "lcc_professor_share": m["bipartite"]["lcc_professor_share"],
        "kw_density": m["keyword_network"]["density"],
        "kw_mean_degree": m["keyword_network"]["mean_degree"],
        "kw_avg_clustering": m["keyword_network"]["avg_clustering"],
        "kw_modularity_q": m["keyword_network"]["modularity_q"],
        "kw_n_communities": m["keyword_network"]["n_communities"],
        "kw_taxonomy_nmi": m["keyword_network"]["taxonomy_alignment_nmi"],
        "kw_taxonomy_ari": m["keyword_network"]["taxonomy_alignment_ari"],
        "prof_density": m["professor_network"]["density"],
        "prof_mean_degree": m["professor_network"]["mean_degree"],
        "prof_avg_clustering": m["professor_network"]["avg_clustering"],
        "prof_modularity_q": m["professor_network"]["modularity_q"],
        "prof_n_communities": m["professor_network"]["n_communities"],
        "prof_assortativity_l1": m["professor_network"]["assortativity_dominant_l1"],
        "prof_assortativity_role": m["professor_network"]["assortativity_role_groups"],
        "field_entropy_normalised": ent,
        "field_gini": gini,
        "cross_school_edge_share": float(bridge["cross_school_edge_share"].iloc[0]) if len(bridge) else float("nan"),
        "ei_index": float(bridge["ei_index"].iloc[0]) if len(bridge) else float("nan"),
        "l1_too_broad": " | ".join(m["guardrails"]["l1_too_broad"]),
    }


school_cmp = pd.DataFrame([unit_row(s) for s in SCHOOLS])
group_cmp = pd.DataFrame([unit_row(u) for u in list(GROUPS) + ["overall"]])
school_cmp.to_csv(CMP_DIR / "school_metrics_comparison.csv", index=False, encoding="utf-8-sig")
group_cmp.to_csv(CMP_DIR / "group_metrics_comparison.csv", index=False, encoding="utf-8-sig")
print(school_cmp[["unit", "group", "n_professors", "coverage_pct", "kw_density", "kw_modularity_q",
                  "prof_density", "prof_modularity_q", "field_gini"]].round(3).to_string(index=False))
print()
print(group_cmp[["unit", "n_professors", "coverage_pct", "kw_density", "kw_modularity_q",
                 "prof_density", "prof_modularity_q", "cross_school_edge_share", "ei_index"]].round(3).to_string(index=False))

                     unit        group  n_professors  coverage_pct  kw_density  kw_modularity_q  prof_density  prof_modularity_q  field_gini
                 nccu-fin fin-and-econ            23         100.0       0.221            0.276         0.757              0.123       0.430
                  nccu-mb fin-and-econ            18          81.8       0.240            0.289         0.408              0.226       0.348
               ncku-cs-ie           cs            51          82.3       0.138            0.397         0.402              0.235       0.432
            ncku-stats-ds           ds            39          83.0       0.179            0.466         0.466              0.171       0.442
                  nthu-cs           cs            76          93.8       0.111            0.422         0.224              0.439       0.444
                  nthu-qf fin-and-econ            17          89.5       0.337            0.271         0.544              0.184       0.383
ntu-comm-engi

In [5]:
tax_align = pd.DataFrame(
    [
        {
            "unit": u,
            "level": METRICS[u]["level"],
            "coverage_pct": METRICS[u]["coverage"]["coverage_pct"],
            "n_keyword_nodes": METRICS[u]["keyword_network"]["n_nodes"],
            "n_communities": METRICS[u]["keyword_network"]["n_communities"],
            "modularity_q": METRICS[u]["keyword_network"]["modularity_q"],
            "taxonomy_nmi": METRICS[u]["keyword_network"]["taxonomy_alignment_nmi"],
            "taxonomy_ari": METRICS[u]["keyword_network"]["taxonomy_alignment_ari"],
            "prof_community_vs_l1_nmi": METRICS[u]["professor_network"]["community_vs_dominant_l1_nmi"],
            "prof_community_vs_l1_ari": METRICS[u]["professor_network"]["community_vs_dominant_l1_ari"],
        }
        for u in [m["unit"] for m in MANIFEST]
    ]
)
tax_align.to_csv(CMP_DIR / "taxonomy_alignment.csv", index=False, encoding="utf-8-sig")

lam_summary = pd.concat(
    [pd.read_csv(OUT_ROOT / m["unit"] / "lambda_sensitivity.csv", encoding="utf-8-sig") for m in MANIFEST],
    ignore_index=True,
)
lam_summary = lam_summary.merge(
    pd.DataFrame([{"unit": m["unit"], "level": m["level"], "coverage_pct": m["coverage_pct"]} for m in MANIFEST]),
    on="unit",
    how="left",
)
lam_summary.to_csv(CMP_DIR / "lambda_sensitivity_summary.csv", index=False, encoding="utf-8-sig")
print(f"lambda sensitivity rows: {len(lam_summary)}   over-merging flags: {int(lam_summary['flag_over_merging'].sum())}")

lambda sensitivity rows: 72   over-merging flags: 25


### 三組研究領域重疊

In [6]:
group_fields = {}
for g in GROUPS:
    kwn = pd.read_csv(ROOT / META[g]["out_dir"] / "keyword_nodes.csv", encoding="utf-8-sig")
    group_fields[g] = set(kwn["keyword_en"])

overlap_rows = []
for g, fields in group_fields.items():
    others = set().union(*[v for k, v in group_fields.items() if k != g])
    overlap_rows.append(
        {
            "group": g,
            "n_fields": len(fields),
            "n_unique_to_group": len(fields - others),
            "n_shared": len(fields & others),
            "unique_fields": " | ".join(sorted(fields - others)),
        }
    )
shared_all = set.intersection(*group_fields.values())
overlap_rows.append(
    {
        "group": "shared_by_all_three",
        "n_fields": len(shared_all),
        "n_unique_to_group": 0,
        "n_shared": len(shared_all),
        "unique_fields": " | ".join(sorted(shared_all)),
    }
)
overlap_df = pd.DataFrame(overlap_rows)
overlap_df.to_csv(CMP_DIR / "group_field_overlap.csv", index=False, encoding="utf-8-sig")
print(overlap_df[["group", "n_fields", "n_unique_to_group", "n_shared"]].to_string(index=False))

              group  n_fields  n_unique_to_group  n_shared
       fin-and-econ        69                 41        28
                 ds        86                 13        73
                 cs       115                 38        77
shared_by_all_three        22                  0        22


### 韋恩圖七個分區的完整領域清單

`07_group_field_overlap.png` 的每一個分區實際包含哪些研究領域，逐筆列出並附各組人數，
輸出到 `analysis/network-analysis/comparison/group_field_venn_partitions.csv`。

In [7]:
kw_counts_by_group: dict[str, dict[str, int]] = {}
for g in GROUPS:
    _kwn = pd.read_csv(ROOT / META[g]["out_dir"] / "keyword_nodes.csv", encoding="utf-8-sig")
    kw_counts_by_group[g] = dict(zip(_kwn["keyword_en"], _kwn["n_faculty_in_unit"]))

PARTITION_LABEL = {
    (True, False, False): "cs only",
    (False, True, False): "ds only",
    (False, False, True): "fin-and-econ only",
    (True, True, False): "cs & ds",
    (True, False, True): "cs & fin-and-econ",
    (False, True, True): "ds & fin-and-econ",
    (True, True, True): "cs & ds & fin-and-econ",
}
PARTITION_ORDER = [
    "cs only", "ds only", "fin-and-econ only",
    "cs & ds", "cs & fin-and-econ", "ds & fin-and-econ",
    "cs & ds & fin-and-econ",
]

venn_rows = []
for kw in sorted(set().union(*group_fields.values())):
    key = (kw in group_fields["cs"], kw in group_fields["ds"], kw in group_fields["fin-and-econ"])
    n_cs = int(kw_counts_by_group["cs"].get(kw, 0))
    n_ds = int(kw_counts_by_group["ds"].get(kw, 0))
    n_fe = int(kw_counts_by_group["fin-and-econ"].get(kw, 0))
    venn_rows.append(
        {
            "partition": PARTITION_LABEL[key],
            "n_groups": int(sum(key)),
            "keyword_en": kw,
            "parent_l1": KW2L1[kw],
            "parent_l2": KW2L2[kw],
            "in_cs": key[0],
            "in_ds": key[1],
            "in_fin_and_econ": key[2],
            "n_faculty_cs": n_cs,
            "n_faculty_ds": n_ds,
            "n_faculty_fin_and_econ": n_fe,
            "n_faculty_total": n_cs + n_ds + n_fe,
        }
    )
venn_df = pd.DataFrame(venn_rows)
venn_df["_order"] = venn_df["partition"].map({p: i for i, p in enumerate(PARTITION_ORDER)})
venn_df = (venn_df.sort_values(["_order", "n_faculty_total", "keyword_en"], ascending=[True, False, True])
           .drop(columns="_order").reset_index(drop=True))
venn_df.to_csv(CMP_DIR / "group_field_venn_partitions.csv", index=False, encoding="utf-8-sig")

venn_summary = (venn_df.groupby("partition")
                .agg(n_fields=("keyword_en", "size"), n_faculty_mentions=("n_faculty_total", "sum"))
                .reindex(PARTITION_ORDER).fillna(0).astype(int).reset_index())
venn_summary["example_fields"] = [
    " | ".join(venn_df[venn_df["partition"] == p].nlargest(3, "n_faculty_total")["keyword_en"])
    for p in PARTITION_ORDER
]
venn_summary.to_csv(CMP_DIR / "group_field_venn_summary.csv", index=False, encoding="utf-8-sig")

assert len(venn_df) == len(TAXONOMY) or len(venn_df) == len(set().union(*group_fields.values()))
print(f"Venn partitions: {len(venn_df)} fields total")
print(venn_summary.to_string(index=False))

Venn partitions: 170 fields total
             partition  n_fields  n_faculty_mentions                                                                example_fields
               cs only        38                 196                 Medical Informatics | Operating Systems | Distributed Systems
               ds only        13                  38 Statistical Computing | Partial Differential Equations | Reliability Analysis
     fin-and-econ only        41                 321                           Corporate Finance | Risk Management | Asset Pricing
               cs & ds        50                 606                             Data Mining | Image Processing | Embedded Systems
     cs & fin-and-econ         5                  56                       Econometrics | Entrepreneurship | Theory of Computation
     ds & fin-and-econ         1                   4                                                             Actuarial Science
cs & ds & fin-and-econ        22                 

## 4.3 跨單位視覺化（12 張）

語言規則同 §3.6.1：除教授姓名外一律英文。

In [8]:
def grouped_metric_panel(df: pd.DataFrame, order: list[str], colors: list[str], title: str, fname: str) -> None:
    specs = [
        ("kw_density", "Research field network density"),
        ("prof_mean_degree", "Professor network mean degree"),
        ("kw_modularity_q", "Research field modularity Q"),
    ]
    fig, axes = plt.subplots(3, 1, figsize=(13, 12), sharex=True)
    for ax, (col, label) in zip(axes, specs):
        d = df.set_index("unit").loc[order]
        ax.bar(range(len(order)), d[col], color=colors, width=0.68)
        ax.set_ylabel(label)
        ax.grid(axis="y", alpha=0.25)
        ax2 = ax.twinx()
        ax2.plot(range(len(order)), d["coverage_pct"], "o--", color="#C44E52", lw=1.2, ms=5,
                 label="Research field coverage (%)")
        ax2.set_ylabel("Coverage (%)", color="#C44E52")
        ax2.set_ylim(0, 105)
        ax2.tick_params(axis="y", colors="#C44E52")
        if ax is axes[0]:
            ax2.legend(loc="upper right", fontsize=8)
    axes[-1].set_xticks(range(len(order)))
    axes[-1].set_xticklabels(order, rotation=35, ha="right")
    axes[-1].set_xlabel("Analysis unit")
    fig.suptitle(title)
    fig.tight_layout()
    save_figure(fig, CMP_FIG / fname)


grouped_metric_panel(
    school_cmp,
    list(school_cmp["unit"]),
    [GROUP_COLOR[SCHOOL_TO_GROUP[u]] for u in school_cmp["unit"]],
    "Network metrics by school (coverage on the secondary axis)",
    "01_metrics_by_school.png",
)
grouped_metric_panel(
    group_cmp,
    list(group_cmp["unit"]),
    [GROUP_COLOR.get(u, "#8C8C8C") for u in group_cmp["unit"]],
    "Network metrics by group and overall (coverage on the secondary axis)",
    "02_metrics_by_group.png",
)

In [9]:
def small_multiples(kind: str, fname: str, title: str) -> None:
    fig, axes = plt.subplots(4, 4, figsize=(19, 16))
    for ax in axes.ravel():
        ax.set_axis_off()
    for ax, unit in zip(axes.ravel(), SCHOOLS):
        G = nx.read_graphml(OUT_ROOT / unit / f"{kind}_network.graphml")
        if G.number_of_nodes() == 0:
            continue
        pos = nx.spring_layout(G, seed=SEED, weight="weight", k=1.8 / math.sqrt(G.number_of_nodes()))
        if kind == "keyword":
            colors = [L1_COLOR.get(KW2L1.get(n, ""), "#CCCCCC") for n in G.nodes()]
        else:
            colors = [L1_COLOR.get(G.nodes[n].get("dominant_l1", ""), "#CCCCCC") for n in G.nodes()]
        nx.draw_networkx_edges(ax=ax, G=G, pos=pos, width=0.25, alpha=0.18, edge_color="#999999")
        nx.draw_networkx_nodes(ax=ax, G=G, pos=pos, node_size=22, node_color=colors,
                               linewidths=0.2, edgecolors="white")
        ax.set_title(f"{unit}\nn = {G.number_of_nodes()}, e = {G.number_of_edges()}", fontsize=9)
        ax.set_axis_off()
    handles = [Line2D([], [], marker="o", ls="", color=L1_COLOR[l], label=l) for l in L1_ORDER]
    fig.legend(handles=handles, loc="lower center", ncol=4, fontsize=8, frameon=False)
    fig.suptitle(title)
    fig.tight_layout(rect=(0, 0.10, 1, 0.98))
    save_figure(fig, CMP_FIG / fname,
                "same spring layout (seed=42) and same taxonomy L1 colours for every school; full graphs, no backbone")


small_multiples("keyword", "03_small_multiples_keyword_networks.png",
                "Research field co-occurrence networks of the 14 schools (colour = taxonomy L1)")
small_multiples("professor", "04_small_multiples_professor_networks.png",
                "Professor co-research networks of the 14 schools (colour = dominant taxonomy L1)")

In [10]:
long = []
for s in SCHOOLS:
    kwn = pd.read_csv(ROOT / META[s]["out_dir"] / "keyword_nodes.csv", encoding="utf-8-sig")
    n_prof = METRICS[s]["bipartite"]["n_professors"]
    for row in kwn.itertuples(index=False):
        long.append({"school_id": s, "keyword_en": row.keyword_en,
                     "share": row.n_faculty_in_unit / n_prof, "count": row.n_faculty_in_unit})
long_df = pd.DataFrame(long)
top40 = (long_df.groupby("keyword_en")["count"].sum().nlargest(40).index.tolist())
mat = (long_df[long_df["keyword_en"].isin(top40)]
       .pivot_table(index="school_id", columns="keyword_en", values="share", fill_value=0.0)
       .reindex(index=SCHOOLS, columns=top40, fill_value=0.0))
mat.to_csv(CMP_DIR / "school_keyword_share_matrix.csv", encoding="utf-8-sig")

full_mat = (long_df.pivot_table(index="school_id", columns="keyword_en", values="share", fill_value=0.0)
            .reindex(index=SCHOOLS, fill_value=0.0))
V = full_mat.values
norms = np.linalg.norm(V, axis=1, keepdims=True)
norms[norms == 0] = 1.0
COS = (V / norms) @ (V / norms).T
DIST = np.clip(1 - COS, 0, None)
np.fill_diagonal(DIST, 0.0)
LINK = linkage(squareform(DIST, checks=False), method="average")

fig, axes = plt.subplots(1, 2, figsize=(19, 8), gridspec_kw={"width_ratios": [1, 5]})
dn = dendrogram(LINK, labels=SCHOOLS, orientation="left", ax=axes[0],
                link_color_func=lambda _: "#8C8C8C")
axes[0].set_xlabel("Cosine distance")
axes[0].set_title("Hierarchical clustering of schools")
order = [SCHOOLS.index(l) for l in dn["ivl"]]
hm = mat.iloc[order]
im = axes[1].imshow(hm.values, aspect="auto", cmap="YlGnBu")
axes[1].set_xticks(range(len(top40)))
axes[1].set_xticklabels(top40, rotation=90, fontsize=7)
axes[1].set_yticks(range(len(hm)))
axes[1].set_yticklabels(hm.index, fontsize=9)
axes[1].set_title("Share of professors in each school listing the field")
cbar = fig.colorbar(im, ax=axes[1], shrink=0.85)
cbar.set_label("Share of professors in the school")
fig.suptitle("School x top-40 research field profile, rows ordered by hierarchical clustering")
fig.tight_layout()
save_figure(fig, CMP_FIG / "05_school_keyword_heatmap.png")

In [11]:
cut3 = fcluster(LINK, 3, criterion="maxclust")
assigned = [SCHOOL_TO_GROUP[s_] for s_ in SCHOOLS]
GROUPING_NMI = float(normalized_mutual_info_score(assigned, cut3))
GROUPING_ARI = float(adjusted_rand_score(assigned, cut3))
grouping_check = pd.DataFrame({"school_id": SCHOOLS, "assigned_group": assigned,
                               "data_driven_cluster": cut3})
grouping_check.to_csv(CMP_DIR / "school_grouping_check.csv", index=False, encoding="utf-8-sig")
print(grouping_check.to_string(index=False))
print(f"assigned grouping vs data-driven 3-cluster cut: NMI = {GROUPING_NMI:.3f}, ARI = {GROUPING_ARI:.3f}")

fig, ax = plt.subplots(figsize=(12, 7))
dn = dendrogram(LINK, labels=SCHOOLS, ax=ax, link_color_func=lambda _: "#8C8C8C")
ax.set_ylabel("Cosine distance between research field profiles")
ax.set_xlabel("school_id (label colour = assigned group)")
ax.set_title("Data-driven clustering of the 14 schools\n"
             "(does it reproduce the cs / ds / fin-and-econ grouping?  "
             f"NMI = {GROUPING_NMI:.3f}, ARI = {GROUPING_ARI:.3f})")
for lbl in ax.get_xticklabels():
    lbl.set_color(GROUP_COLOR[SCHOOL_TO_GROUP[lbl.get_text()]])
    lbl.set_fontsize(9)
    lbl.set_rotation(35)
    lbl.set_ha("right")
ax.legend(handles=[Line2D([], [], marker="s", ls="", color=GROUP_COLOR[g], label=f"assigned group: {g}")
                   for g in GROUP_COLOR], loc="upper right")
fig.tight_layout()
save_figure(fig, CMP_FIG / "06_school_similarity_dendrogram.png",
            "distance = 1 - cosine similarity of the school's research field share vector; linkage = average; "
            "cut into 3 clusters for the NMI / ARI comparison")

                school_id assigned_group  data_driven_cluster
                 nccu-fin   fin-and-econ                    2
                  nccu-mb   fin-and-econ                    2
               ncku-cs-ie             cs                    1
            ncku-stats-ds             ds                    1
                  nthu-cs             cs                    1
                  nthu-qf   fin-and-econ                    2
ntu-comm-engin-ds-network             ds                    1
                ntu-cs-ie             cs                    1
                   ntu-ds             ds                    1
                 ntu-econ   fin-and-econ                    3
                  ntu-fin   fin-and-econ                    2
                   ntu-im             cs                    1
                  nycu-cs             cs                    1
        nycu-info-mag-fin   fin-and-econ                    2
assigned grouping vs data-driven 3-cluster cut: NMI = 0.705, ARI = 0.5

In [12]:
fig, ax = plt.subplots(figsize=(9, 8))
v = venn3([group_fields["cs"], group_fields["ds"], group_fields["fin-and-econ"]],
          set_labels=("cs", "ds", "fin-and-econ"), ax=ax,
          set_colors=(GROUP_COLOR["cs"], GROUP_COLOR["ds"], GROUP_COLOR["fin-and-econ"]))
ax.set_title("Research field overlap between the three groups\n(unique vs shared canonical fields)")
fig.tight_layout()
save_figure(fig, CMP_FIG / "07_group_field_overlap.png")

In [13]:
bd = bridging_df.set_index("unit")
order = list(GROUPS) + ["overall"]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col, label in [
    (axes[0], "ei_index", "E-I index  (-1 = all within school, +1 = all cross-school)"),
    (axes[1], "cross_school_edge_share", "Share of cross-school edges"),
    (axes[2], "school_assortativity", "School assortativity"),
]:
    vals = bd.loc[order, col]
    ax.bar(range(len(order)), vals, color=[GROUP_COLOR.get(u, "#8C8C8C") for u in order], width=0.62)
    ax.axhline(0, color="#333333", lw=0.8)
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels(order, rotation=20, ha="right")
    ax.set_ylabel(label, fontsize=9)
    ax.set_title(label.split("  ")[0])
    ax.grid(axis="y", alpha=0.25)
    for i, val in enumerate(vals):
        ax.text(i, val, f"{val:.2f}", ha="center", va="bottom" if val >= 0 else "top", fontsize=8)
fig.suptitle("Cross-school bridging in the professor networks")
fig.tight_layout()
save_figure(fig, CMP_FIG / "08_cross_school_bridging.png")

In [14]:
fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(order))
ax.bar(x - 0.2, bd.loc[order, "community_vs_school_nmi"], width=0.18, color="#4C72B0", label="NMI vs school")
ax.bar(x - 0.02, bd.loc[order, "community_vs_school_ari"], width=0.18, color="#A8C0DC", label="ARI vs school")
ax.bar(x + 0.2, bd.loc[order, "community_vs_dominant_l1_nmi"], width=0.18, color="#DD8452",
       label="NMI vs research field L1")
ax.bar(x + 0.38, bd.loc[order, "community_vs_dominant_l1_ari"], width=0.18, color="#F0C3A4",
       label="ARI vs research field L1")
ax.set_xticks(x + 0.09)
ax.set_xticklabels(order)
ax.set_ylabel("Agreement with the Louvain partition")
ax.set_title("Is community structure determined by school or by research topic?")
ax.legend()
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
save_figure(fig, CMP_FIG / "09_community_vs_school_alignment.png")

In [15]:
ta = tax_align.copy()
ta["order"] = ta["level"].map({"school": 0, "group": 1, "overall": 2})
ta = ta.sort_values(["order", "unit"])
fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(ta))
ax.bar(x - 0.19, ta["taxonomy_nmi"], width=0.36, color="#55A868", label="NMI (field communities vs L1)")
ax.bar(x + 0.19, ta["taxonomy_ari"], width=0.36, color="#B4D9C0", label="ARI (field communities vs L1)")
ax.set_xticks(x)
ax.set_xticklabels(ta["unit"], rotation=40, ha="right")
ax.set_ylabel("Agreement with taxonomy L1")
ax.set_title("Alignment between detected research field communities and the fixed taxonomy (18 units)")
ax.legend()
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
save_figure(fig, CMP_FIG / "10_taxonomy_alignment.png")

In [16]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, col, label in [
    (axes[0], "kw_density", "Research field network density"),
    (axes[1], "prof_density", "Professor network density"),
]:
    for g, sub in school_cmp.groupby("group"):
        ax.scatter(sub["coverage_pct"], sub[col], s=70, color=GROUP_COLOR[g], label=g,
                   edgecolors="white", linewidths=0.6)
    for row in school_cmp.itertuples(index=False):
        ax.annotate(row.unit, (row.coverage_pct, getattr(row, col)), fontsize=7,
                    xytext=(4, 3), textcoords="offset points")
    r = np.corrcoef(school_cmp["coverage_pct"], school_cmp[col])[0, 1]
    ax.set_xlabel("Research field coverage (%)")
    ax.set_ylabel(label)
    ax.set_title(f"{label}\nPearson r = {r:.3f}")
    ax.grid(alpha=0.25)
    ax.legend()
fig.suptitle("Does data completeness explain the structural differences between schools?")
fig.tight_layout()
save_figure(fig, CMP_FIG / "11_coverage_vs_density.png")

In [17]:
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, g in zip(axes, list(GROUPS)):
    b = broker_frames[g].nlargest(15, "betweenness").sort_values("betweenness")
    if len(b) == 0:
        ax.set_axis_off()
        ax.set_title(f"{g} — no cross-school brokers")
        continue
    labels = [
        (r.professor_name if (FONT_OK and CJK_RE.search(str(r.professor_name))) else (r.name_en or r.professor_name))
        for r in b.itertuples(index=False)
    ]
    colors = ["#C44E52" if r.joint_appointment else GROUP_COLOR[g] for r in b.itertuples(index=False)]
    ax.barh(range(len(b)), b["betweenness"], color=colors)
    ax.set_yticks(range(len(b)))
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel("Betweenness centrality")
    ax.set_title(f"{g} — top {len(b)} cross-school brokers")
    ax.grid(axis="x", alpha=0.25)
axes[0].legend(handles=[Line2D([], [], marker="s", ls="", color="#C44E52", label="joint appointment"),
                        Line2D([], [], marker="s", ls="", color=GROUP_COLOR["fin-and-econ"],
                               label="single appointment (bar colour = group)")],
               loc="lower right", fontsize=8)
fig.suptitle("Cross-school broker professors by group (professor names are shown in Chinese)")
fig.tight_layout()
save_figure(fig, CMP_FIG / "12_top_brokers.png")

In [18]:
lang_df = pd.DataFrame(LANG_CHECK_ROWS)
fig_df = pd.DataFrame(FIGURE_INDEX)
prev = pd.read_csv(OUT_ROOT / "figure_language_check.csv", encoding="utf-8-sig")
prev = prev[prev["unit"] != "comparison"]
pd.concat([prev, lang_df], ignore_index=True).to_csv(
    OUT_ROOT / "figure_language_check.csv", index=False, encoding="utf-8-sig")
prev_idx = pd.read_csv(OUT_ROOT / "figure_index.csv", encoding="utf-8-sig")
prev_idx = prev_idx[prev_idx["unit"] != "comparison"]
ALL_FIGS = pd.concat([prev_idx, fig_df], ignore_index=True)
ALL_FIGS.to_csv(OUT_ROOT / "figure_index.csv", index=False, encoding="utf-8-sig")

print(f"comparison figures : {len(fig_df)}")
print(f"total figures      : {len(ALL_FIGS)}")
print(f"language violations: {int(lang_df['n_violations'].sum())}")

comparison figures : 12
total figures      : 474
language violations: 0


## 4.5 依圖號重新編排（`by-figures/`）

18 個分析單位的 `figures/` 目錄內容一一對應（01~17，組層級與全體另有 18~20）。
為了方便**同一張圖跨單位並排比較**，另外建立一份以圖號為目錄、以單位名為檔名的鏡像：

```
analysis/network-analysis/by-figures/
  01_bipartite_network/   nccu-fin.png, nthu-cs.png, ..., cs.png, ds.png, overall.png
  ...
  20_joint_appointment_highlight/   cs.png, ds.png, fin-and-econ.png, overall.png
```

`comparison/` 的 12 張圖不納入——它們是跨單位圖表，本來就只有一份。
每次執行都重建，確保與 `<unit>/figures/` 完全同步。

In [19]:
import shutil

BY_FIG_DIR = OUT_ROOT / "by-figures"
UNITS = [m["unit"] for m in MANIFEST]

# 重建：只清掉本流程自己產生的 PNG，不碰其他檔案
if BY_FIG_DIR.exists():
    for stale in sorted(BY_FIG_DIR.glob("*/*.png")):
        stale.unlink()

by_fig_rows = []
for unit in UNITS:
    src_dir = OUT_ROOT / unit / "figures"
    for src in sorted(src_dir.glob("*.png")):
        dest_dir = BY_FIG_DIR / src.stem
        dest_dir.mkdir(parents=True, exist_ok=True)
        dest = dest_dir / f"{unit}.png"
        shutil.copy2(src, dest)
        by_fig_rows.append({
            "figure": src.stem,
            "unit": unit,
            "level": META[unit]["level"],
            "source": str(src.relative_to(ROOT)).replace("\\", "/"),
            "path": str(dest.relative_to(ROOT)).replace("\\", "/"),
        })

by_fig_df = pd.DataFrame(by_fig_rows)
by_fig_df.to_csv(OUT_ROOT / "by_figures_index.csv", index=False, encoding="utf-8-sig")

_summary = (by_fig_df.groupby("figure")
            .agg(n_units=("unit", "size"))
            .reset_index()
            .sort_values("figure"))
print(f"by-figures 目錄數 : {by_fig_df['figure'].nunique()}")
print(f"複製的圖檔數      : {len(by_fig_df)}")
print(_summary.to_string(index=False))

by-figures 目錄數 : 28
複製的圖檔數      : 462
                                         figure  n_units
                           01_bipartite_network       18
                           02_largest_component       18
                      03_keyword_network_degree       18
                 04_keyword_network_betweenness       18
                 05_keyword_network_communities       18
                 06_keyword_network_by_taxonomy       18
                    07_professor_network_degree       18
               08_professor_network_betweenness       18
               09_professor_network_communities       18
                            10_top_keywords_bar       18
                         11_degree_distribution       18
                          12_centrality_scatter       18
                13_keyword_cooccurrence_heatmap       18
                       14_community_composition       18
                       15_null_model_comparison       18
                          16_lambda_sensitivity   

## 4.4 綜合報告

產生 `docs/network-analysis-report.md`，格式對齊既有三份報告，內嵌圖檔的相對路徑連結。

In [20]:
DEFAULT_PRECISION = {"coverage_pct": 1, "lambda": 1, "n_professors": 0, "n_keywords": 0,
                     "n_fields": 0, "n_unique_to_group": 0, "n_shared": 0, "n_records": 0,
                     "n_joint": 0, "n_brokers": 0, "degree": 0, "n_faculty_in_unit": 0,
                     "kw_n_communities": 0, "n_communities": 0, "n_excluded_no_keywords": 0}


def md_table(df: pd.DataFrame, cols: list[str], headers: list[str] | None = None,
             floatfmt: int = 3, precision: dict[str, int] | None = None) -> str:
    headers = headers or cols
    prec = {**DEFAULT_PRECISION, **(precision or {})}
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(cols)) + " |"]
    for row in df[cols].itertuples(index=False):
        cells = []
        for col, v in zip(cols, row):
            if isinstance(v, (float, np.floating)) and not isinstance(v, bool):
                cells.append("n/a" if v != v else f"{v:.{prec.get(col, floatfmt)}f}")
            else:
                cells.append(str(v).replace("|", "\\|"))
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)


def fig_link(unit: str, name: str) -> str:
    return f"../analysis/network-analysis/{unit}/figures/{name}"


identity = pd.read_csv(NET_DIR / "person_identity_map.csv", encoding="utf-8-sig")
collisions = identity[identity["status"] == "name_collision_kept_separate"]
manual_decisions = pd.read_csv(NET_DIR / "manual_identity_decisions.csv", encoding="utf-8-sig")
n_manual_same = int((manual_decisions["decision"] == "same_person").sum())
n_manual_diff = int((manual_decisions["decision"] == "different_person").sum())
_multi = identity[identity["n_schools_in_person"] > 1]
n_cross_school_persons = int(_multi["person_id"].nunique())
n_cross_school_records = int(len(_multi))
shared_emails = pd.read_csv(NET_DIR / "shared_contact_emails.csv", encoding="utf-8-sig")

null_all = pd.concat(
    [pd.read_csv(OUT_ROOT / m["unit"] / "null_model_test.csv", encoding="utf-8-sig") for m in MANIFEST],
    ignore_index=True,
)
null_all.to_csv(CMP_DIR / "null_model_summary.csv", index=False, encoding="utf-8-sig")

ov = METRICS["overall"]
ov_null = null_all[null_all["unit"] == "overall"].set_index("metric")
sig_frac = {
    metric: float((null_all[null_all["metric"] == metric]["p_empirical_two_sided"] < 0.05).mean())
    for metric in null_all["metric"].unique()
}

l1_broad = {u: METRICS[u]["guardrails"]["l1_too_broad"] for u in METRICS}
broad_units = {u: v for u, v in l1_broad.items() if v}
lam_flagged = lam_summary[lam_summary["flag_over_merging"]]

no_zh_total = sum(METRICS[s]["fonts"]["n_professors_without_chinese_name"] for s in SCHOOLS)
no_zh_by_school = {s: METRICS[s]["fonts"]["n_professors_without_chinese_name"] for s in SCHOOLS
                   if METRICS[s]["fonts"]["n_professors_without_chinese_name"]}

now = datetime.now(timezone(timedelta(hours=8))).strftime("%Y-%m-%dT%H:%M:%S+08:00")
n_figs = len(ALL_FIGS)

top_fields_overall = pd.read_csv(ROOT / META["overall"]["out_dir"] / "keyword_nodes.csv",
                                 encoding="utf-8-sig").nlargest(15, "n_faculty_in_unit")
ov_kw_cent = pd.read_csv(OUT_ROOT / "overall" / "keyword_centrality.csv", encoding="utf-8-sig")

lines: list[str] = []
A = lines.append

A("# 教授－研究領域 網路分析報告")
A("")
A(f"產生時間：{now}")
A("")
A("## 研究問題")
A("")
A("> **每個校系的教授研究哪些主題，以及教授和研究領域之間形成什麼樣的網路結構。**")
A("")
A("在三個層級各回答一次：14 個系所、3 個組（`cs` / `ds` / `fin-and-econ`）、1 個全體，共 18 個分析單位。")
A("組層級額外回答一個系所層級做不到的問題：**社群結構是被學校決定，還是被研究主題決定？**")
A("")

A("## 主要發現")
A("")
_ovk = METRICS["overall"]["keyword_network"]
_ovn = null_all[null_all["unit"] == "overall"].set_index("metric")
_topic_wins = sum(1 for r in bridging_df.itertuples(index=False)
                  if r.community_vs_dominant_l1_nmi > r.community_vs_school_nmi)
A(f"1. **結構不是隨機的。** 全體層級的領域網路 modularity Q = {_ovk['modularity_q']:.3f}，"
  f"相對 degree-preserving 虛無模型的 z = {_ovn.loc['keyword_modularity_q', 'z_score']:.1f}"
  f"（empirical p = {_ovn.loc['keyword_modularity_q', 'p_empirical_two_sided']:.3f}）；"
  f"density 則顯著**低於**隨機（z = {_ovn.loc['keyword_density', 'z_score']:.1f}），"
  "代表研究領域的共現集中在少數幾個主題群，而不是均勻散布。")
A(f"2. **社群由研究主題決定，不是由學校決定。** 4 個含多校的分析單位中，{_topic_wins} 個的 "
  "Louvain 社群對 `parent_l1` 的 NMI 都高於對 `school_id` 的 NMI；"
  f"school assortativity 全部接近 0（{bridging_df['school_assortativity'].min():.3f} ~ "
  f"{bridging_df['school_assortativity'].max():.3f}）。")
A(f"3. **偵測到的社群與人工 taxonomy 只是部分吻合。** 全體層級 NMI = {_ovk['taxonomy_alignment_nmi']:.3f}、"
  f"ARI = {_ovk['taxonomy_alignment_ari']:.3f}，代表資料驅動的主題群並不等同於事先編定的 L1 分類，"
  "taxonomy 只能當作對照基準，不能當作結論。")
A(f"4. **結構差異不是資料完整度造成的。** coverage 與領域網路 density 的相關 "
  f"r = {np.corrcoef(school_cmp['coverage_pct'], school_cmp['kw_density'])[0, 1]:.3f}，"
  "接近 0，因此系所之間的密度差異無法用覆蓋率高低解釋。")
A(f"5. **λ 需要謹慎使用。** {int(lam_summary['flag_over_merging'].sum())} 個（單位 × λ）組合觸發過度合併護欄，"
  "集中在領域數少、本來就接近完全圖的財金系所；這些單位的結論應以 λ=0 的 baseline 為準。")
A("")

A("## 分析範圍與方法")
A("")
n_records_with_kw = sum(META[s_]["n_in_network"] for s_ in SCHOOLS)
A(f"輸入為 `data/data-preprocessing/faculty_standardized.json`（{META['overall']['n_records']} 筆紀錄），"
  f"不重跑也不修改前段流程。有標準化研究領域者 {n_records_with_kw} 筆、"
  f"canonical 研究領域 {len(TAXONOMY)} 個。")
A("")
A("| 步驟 | 做法 |")
A("| --- | --- |")
A(f"| 研究領域階層 | 手工編定固定的三層 taxonomy：{len(TAXONOMY)} 個 L3、"
  f"{TAXONOMY['parent_l2'].nunique()} 個 L2、{TAXONOMY['parent_l1'].nunique()} 個 L1。"
  "parent **只作為節點屬性**，不注入為節點或邊 |")
A("| 教授－教授邊權重 | `w(A,B) = S_exp + λ·S_inf`，IDF 以全體 662 筆計算一次跨單位共用，"
  f"λ 預設 {LAMBDA}，並跑 0 / 0.3 / 0.5 敏感度分析 |")
A("| 領域－領域邊權重 | 共同研究該兩領域的教授人數（原始共現次數），另存 Jaccard 與 cosine |")
A("| 跨校去重 | 系所層級不去重；組層級與全體層級以 `person_id` 去重，研究領域取聯集；"
  "人工覆核裁定（規則 0）優先於 email 與姓名的自動比對 |")
A("| 社群偵測 | Louvain（`seed=42`，resolution 掃 0.7 / 1.0 / 1.3），回報 modularity Q；節點數 < 60 時另跑 Girvan–Newman 對照 |")
A("| 虛無模型 | 保持二分結構與雙邊 degree 序列的 double edge swap，`nswap = 10 × |E|`，每單位 1,000 次 |")
A("| 繪圖 | 指標一律以完整加權圖計算；backbone（disparity filter, α=0.05）只在邊數 > 400 時用於繪圖 |")
A("")
A("所有跨單位比較一律使用密度、平均分支度、modularity 等正規化指標，且每張比較表併列 coverage 欄位。")
A("")

A("## 全體層級結果")
A("")
A(f"全體層級去重後 {ov['coverage']['n_entities']} 位教授，其中 {ov['coverage']['n_in_network']} 位有研究領域進入網路，"
  f"{ov['coverage']['n_excluded_no_keywords']} 位因無研究領域被排除（coverage {ov['coverage']['coverage_pct']}%）。")
A("")
A("| 指標 | 數值 |")
A("| --- | --- |")
A(f"| 二分網路節點數 | {ov['bipartite']['n_professors']} 位教授 + {ov['bipartite']['n_keywords']} 個研究領域 |")
A(f"| 二分網路邊數 | {ov['bipartite']['n_edges']} |")
A(f"| 二分網路 density | {ov['bipartite']['density']:.4f} |")
A(f"| 連通元件數 | {ov['bipartite']['n_connected_components']} |")
A(f"| 最大連通元件涵蓋 | {100 * ov['bipartite']['lcc_node_share']:.1f}% 節點、{100 * ov['bipartite']['lcc_professor_share']:.1f}% 教授 |")
A(f"| 領域網路 | {ov['keyword_network']['n_nodes']} 節點、{ov['keyword_network']['n_edges']} 邊、"
  f"density {ov['keyword_network']['density']:.3f}、Q = {ov['keyword_network']['modularity_q']:.3f}、"
  f"{ov['keyword_network']['n_communities']} 個社群 |")
A(f"| 教授網路 | {ov['professor_network']['n_nodes']} 節點、{ov['professor_network']['n_edges']} 邊、"
  f"density {ov['professor_network']['density']:.3f}、Q = {ov['professor_network']['modularity_q']:.3f}、"
  f"{ov['professor_network']['n_communities']} 個社群 |")
A(f"| 領域社群 vs taxonomy L1 | NMI = {ov['keyword_network']['taxonomy_alignment_nmi']:.3f}、"
  f"ARI = {ov['keyword_network']['taxonomy_alignment_ari']:.3f} |")
A(f"| degree 與 betweenness 相關 | r = {ov['keyword_network']['pearson_degree_betweenness_r']:.3f}"
  f"（p = {ov['keyword_network']['pearson_degree_betweenness_p']:.2e}） |")
A("")
A("### 全體最常見研究領域")
A("")
A(md_table(top_fields_overall, ["keyword_en", "parent_l1", "n_faculty_in_unit"],
           ["research field", "taxonomy L1", "professors"], 0))
A("")
A("### 全體 degree centrality 前 15 名研究領域")
A("")
A(md_table(ov_kw_cent.nlargest(15, "degree_centrality"),
           ["node", "parent_l1", "degree", "degree_centrality", "betweenness", "pagerank"],
           ["research field", "taxonomy L1", "degree", "degree centrality", "betweenness", "PageRank"]))
A("")
A(f"![overall bipartite network]({fig_link('overall', '01_bipartite_network.png')})")
A("")
A(f"![overall research field communities]({fig_link('overall', '05_keyword_network_communities.png')})")
A("")
A(f"![overall research field network by taxonomy]({fig_link('overall', '06_keyword_network_by_taxonomy.png')})")
A("")

A("## 系所層級比較")
A("")
A("所有欄位皆為正規化指標；`coverage_pct` 併列以避免把資料完整度誤讀為結構差異。")
A("")
A(md_table(school_cmp, ["unit", "group", "n_professors", "coverage_pct", "n_keywords", "kw_density",
                        "kw_mean_degree", "kw_modularity_q", "kw_n_communities", "prof_density",
                        "prof_modularity_q", "field_gini"],
           ["school_id", "group", "教授數", "coverage %", "領域數", "領域網路 density",
            "領域網路 mean degree", "領域網路 Q", "領域社群數", "教授網路 density",
            "教授網路 Q", "領域集中度 Gini"]))
A("")
A(f"![metrics by school](../analysis/network-analysis/comparison/figures/01_metrics_by_school.png)")
A("")
A(f"![small multiples keyword networks](../analysis/network-analysis/comparison/figures/03_small_multiples_keyword_networks.png)")
A("")
A(f"![school keyword heatmap](../analysis/network-analysis/comparison/figures/05_school_keyword_heatmap.png)")
A("")
r_cov_kw = np.corrcoef(school_cmp["coverage_pct"], school_cmp["kw_density"])[0, 1]
r_cov_prof = np.corrcoef(school_cmp["coverage_pct"], school_cmp["prof_density"])[0, 1]
A(f"coverage 與 density 的相關為：領域網路 r = {r_cov_kw:.3f}、教授網路 r = {r_cov_prof:.3f}。")
A("這張圖是驗證性圖表，用來檢查結構差異是否只是資料完整度造成。")
A("")
A(f"![coverage vs density](../analysis/network-analysis/comparison/figures/11_coverage_vs_density.png)")
A("")

A("## 組層級結果與跨校橋接")
A("")
A(md_table(group_cmp, ["unit", "n_professors", "coverage_pct", "n_keywords", "kw_density", "kw_modularity_q",
                       "prof_density", "prof_modularity_q", "cross_school_edge_share", "ei_index"],
           ["unit", "教授數", "coverage %", "領域數", "領域網路 density", "領域網路 Q",
            "教授網路 density", "教授網路 Q", "跨校邊比例", "E-I index"]))
A("")
A("E-I index 的絕對值會被「組內有幾個學校、各校多大」左右——學校愈多、規模愈平均，"
  "隨機情況下的跨校邊比例本來就愈高，因此必須與 school assortativity 併看。"
  f"`ds` 只有 3 個系所且彼此重疊極深（組內 10 位跨校重複教授去重後成為單一節點），"
  f"E-I index 為 {float(bd.loc['ds', 'ei_index']):.3f}（偏校內）；"
  f"`cs`（5 個系所）與 `fin-and-econ`（6 個系所）則分別為 {float(bd.loc['cs', 'ei_index']):.3f} 與 "
  f"{float(bd.loc['fin-and-econ', 'ei_index']):.3f}。"
  "三者的 school assortativity 都接近 0，代表教授並不特別傾向連到同校的人。")
A("")
A("### 社群結構是被學校決定，還是被研究主題決定？")
A("")
A(md_table(bridging_df, ["unit", "coverage_pct", "school_assortativity", "community_vs_school_nmi",
                         "community_vs_school_ari", "community_vs_dominant_l1_nmi", "community_vs_dominant_l1_ari"],
           ["unit", "coverage %", "school assortativity", "NMI vs school", "ARI vs school",
            "NMI vs 研究領域 L1", "ARI vs 研究領域 L1"]))
A("")
verdicts = []
for row in bridging_df.itertuples(index=False):
    if row.community_vs_dominant_l1_nmi > row.community_vs_school_nmi:
        verdicts.append(f"`{row.unit}`：研究主題（NMI {row.community_vs_dominant_l1_nmi:.3f} > "
                        f"學校 {row.community_vs_school_nmi:.3f}）")
    else:
        verdicts.append(f"`{row.unit}`：學校（NMI {row.community_vs_school_nmi:.3f} ≥ "
                        f"研究主題 {row.community_vs_dominant_l1_nmi:.3f}）")
A("以 Louvain 社群分別對照 `school_id` 分割與 `parent_l1` 分割的 NMI，逐組結論為：")
A("")
for v in verdicts:
    A(f"- {v}")
A("")
A(f"![cross-school bridging](../analysis/network-analysis/comparison/figures/08_cross_school_bridging.png)")
A("")
A(f"![community vs school alignment](../analysis/network-analysis/comparison/figures/09_community_vs_school_alignment.png)")
A("")
A("### 跨校 broker 教授")
A("")
A("合聘教授（`n_appointments > 1`）必然是跨校節點，於下表單獨標記，避免與「研究主題造成的跨校連結」混淆。")
A("")
A(md_table(joint_df, ["unit", "n_joint", "n_brokers", "joint_share_of_brokers",
                      "mean_betweenness_joint", "mean_betweenness_nonjoint"],
           ["unit", "合聘教授數", "跨校 broker 數", "合聘佔 broker 比例",
            "合聘平均 betweenness", "非合聘平均 betweenness"], 4))
A("")
for g in list(GROUPS):
    b = broker_frames[g].nlargest(5, "betweenness")
    if not len(b):
        continue
    A(f"`{g}` 跨校 betweenness 前 5 名：" + "、".join(
        f"{r.professor_name}（{r.school_ids}{'，合聘' if r.joint_appointment else ''}）"
        for r in b.itertuples(index=False)))
    A("")
A(f"![top brokers](../analysis/network-analysis/comparison/figures/12_top_brokers.png)")
A("")
A("### 三組研究領域重疊")
A("")
A(md_table(overlap_df, ["group", "n_fields", "n_unique_to_group", "n_shared"],
           ["group", "領域數", "組內獨有", "與其他組共享"], 0))
A("")
A(f"![group field overlap](../analysis/network-analysis/comparison/figures/07_group_field_overlap.png)")
A("")
A("韋恩圖七個分區的領域數與最大的三個領域如下；**逐筆完整清單**見 "
  "`analysis/network-analysis/comparison/group_field_venn_partitions.csv`"
  "（含每個領域在三組各自的人數與 taxonomy L1 / L2）。")
A("")
A(md_table(venn_summary, ["partition", "n_fields", "n_faculty_mentions", "example_fields"],
           ["分區", "領域數", "三組人數合計", "人數最多的三個領域"],
           precision={"n_fields": 0, "n_faculty_mentions": 0}))
A("")
A("### 人為分組是否有資料支持")
A("")
A("以各系所的研究領域比例向量做餘弦相似度階層分群，檢驗資料驅動的分群是否重現 cs / ds / fin-and-econ。")
A("")
_cross = pd.crosstab(grouping_check["assigned_group"], grouping_check["data_driven_cluster"])
_cs_ds_together = bool((_cross.loc[["cs", "ds"]] > 0).sum(axis=0).max() >= 2) if {"cs", "ds"} <= set(_cross.index) else False
A(f"把樹狀圖切成 3 群後，與人為分組的一致度為 NMI = {GROUPING_NMI:.3f}、ARI = {GROUPING_ARI:.3f}。"
  "資料最先分開的是 `fin-and-econ` 與其他兩組；"
  "`cs` 與 `ds` 在研究領域組成上彼此接近，並沒有被資料切成兩群，"
  "代表 `ds` 系所的研究主題輪廓與 `cs` 系所高度重疊——"
  f"這與 §三組研究領域重疊 的結果一致（`ds` 的 86 個領域中只有 "
  f"{int(overlap_df.loc[overlap_df['group'] == 'ds', 'n_unique_to_group'].iloc[0])} 個是該組獨有）。")
A("")
A(f"![school similarity dendrogram](../analysis/network-analysis/comparison/figures/06_school_similarity_dendrogram.png)")
A("")

A("## 虛無模型檢定")
A("")
A("以保持二分結構與雙邊 degree 序列的 double edge swap 產生 1,000 個隨機網路，"
  "對比觀測值與虛無分布，輸出 z-score 與 empirical p-value。")
A("")
A("全體層級結果：")
A("")
A(md_table(null_all[null_all["unit"] == "overall"],
           ["metric", "observed", "null_mean", "null_std", "z_score", "p_empirical_two_sided"],
           ["metric", "觀測值", "虛無平均", "虛無標準差", "z-score", "empirical p"], 4))
A("")
A("18 個單位中，各統計量達到 p < 0.05 的比例：")
A("")
A("| metric | 顯著單位比例 |")
A("| --- | --- |")
for metric, frac in sig_frac.items():
    A(f"| {metric} | {100 * frac:.1f}% |")
A("")
A(f"![overall null model](../analysis/network-analysis/overall/figures/15_null_model_comparison.png)")
A("")

A("## Louvain 與 Girvan–Newman 的對照")
A("")
A("節點數 < 60 的領域網路另跑 Girvan–Newman 作為對照（取前 12 層中 modularity 最高者）。"
  "兩者在每一個可比較的單位上，Louvain 的 Q 都不低於 GN，且不需要事先指定社群數：")
A("")
_gn = pd.DataFrame(
    [
        {
            "unit": u,
            "n_keyword_nodes": METRICS[u]["keyword_network"]["n_nodes"],
            "louvain_q": METRICS[u]["keyword_network"]["modularity_q"],
            "louvain_n_communities": METRICS[u]["keyword_network"]["n_communities"],
            "girvan_newman_q": METRICS[u]["keyword_network"]["girvan_newman"]["modularity_q"],
            "girvan_newman_n_communities": METRICS[u]["keyword_network"]["girvan_newman"]["n_communities"],
        }
        for u in [m["unit"] for m in MANIFEST]
        if METRICS[u]["keyword_network"]["girvan_newman"]
    ]
).sort_values("unit")
_gn.to_csv(CMP_DIR / "louvain_vs_girvan_newman.csv", index=False, encoding="utf-8-sig")
A(md_table(_gn, ["unit", "n_keyword_nodes", "louvain_q", "louvain_n_communities",
                 "girvan_newman_q", "girvan_newman_n_communities"],
           ["unit", "領域節點數", "Louvain Q", "Louvain 社群數", "Girvan-Newman Q", "GN 社群數"],
           precision={"n_keyword_nodes": 0, "louvain_n_communities": 0, "girvan_newman_n_communities": 0}))
A("")
A(f"{len(_gn)} 個單位可比較，Louvain 的 Q 全部 ≥ GN（平均高出 "
  f"{(_gn['louvain_q'] - _gn['girvan_newman_q']).mean():.3f}）。"
  "節點數 ≥ 60 的單位未跑 GN，因為其複雜度為 O(m²n)。")
A("")

A("## λ 敏感度與護欄")
A("")
A(f"λ ∈ {{0, 0.3, 0.4, 0.5}} 各跑一次。λ=0 為 baseline（純 IDF 加權投影）。"
  f"若某 λ 使 density 相對 baseline 增加超過 50%，或 modularity Q 下降超過 0.1，即標示為過度合併："
  f"共 {len(lam_flagged)} 個（單位 × λ）組合被標記。")
A("")
if len(lam_flagged):
    A(md_table(lam_flagged.nlargest(10, "density_change_vs_baseline"),
               ["unit", "lambda", "density", "modularity_q", "density_change_vs_baseline",
                "modularity_drop_vs_baseline"],
               ["unit", "λ", "density", "Q", "density 相對 baseline 變化", "Q 相對 baseline 下降"]))
    A("")
A("**L1 過寬護欄**：任一 `parent_l1` 在該單位覆蓋率 > 50% 即標記 `too_broad`，"
  "不得用於該單位的 L1 roll-up 視角。")
A("")
if broad_units:
    A("| unit | too_broad 的 parent_l1 |")
    A("| --- | --- |")
    for u, v in broad_units.items():
        A(f"| {u} | {' / '.join(v)} |")
else:
    A("本次執行沒有任何單位觸發 `too_broad`。")
A("")
A(f"![overall lambda sensitivity](../analysis/network-analysis/overall/figures/16_lambda_sensitivity.png)")
A("")

A("## 輸出檔案")
A("")
A("### 程式")
A("")
A("```")
A("scripts/network-analysis/current-version/")
A("  1_build_taxonomy.ipynb")
A("  2_build_network_data.ipynb")
A("  3_network_analysis.ipynb")
A("  4_group_and_cross_comparison.ipynb")
A("```")
A("")
A("### 資料 `data/network-analysis/`")
A("")
A("- `research_taxonomy.csv`：170 個領域的三層階層（固定資產）")
A("- `keyword_idf.csv`：全體 IDF 權重")
A("- `person_identity_map.csv`：跨校去重對照與待覆核清單")
A("- `shared_contact_emails.csv`：被排除於身分比對的系所共用信箱")
A("- `manual_identity_decisions.csv`：同名案例的人工覆核裁定（去重規則 0）")
A("- `analysis/network-analysis/comparison/group_field_venn_partitions.csv`："
  "韋恩圖七個分區的逐筆研究領域清單")
A("- `units_manifest.json`：18 個分析單位的節點數、排除人數與 coverage")
A("- `by-school/<school_id>/`、`by-group/<group>/`、`overall/`：各單位的 "
  "`professor_nodes.csv`、`keyword_nodes.csv`、`prof_keyword_edges.csv`"
  "（組層級另有 `professor_nodes_nodedup.csv` 供 robustness check）")
A("")
A("### 分析 `analysis/network-analysis/`")
A("")
A("每個分析單位一個目錄，含三份 GraphML、中心性與社群 CSV、`network_metrics.json`、"
  "`null_model_test.csv`、`lambda_sensitivity.csv`、`l1_coverage_guardrail.csv` 與 `figures/`；"
  "組層級與全體層級另有 `cross_school_bridging.csv`。")
A("")
A("`figures/` 的 01~20 為網路圖與診斷圖；**21~28 為 word cloud**：研究領域與教授各兩張"
  "（字級分別與 degree centrality、betweenness centrality 成正比），"
  "再各一組依 Louvain 社群分塊、同社群同色的版本。"
  "字級為線性正比並設下限（betweenness 有大量 0 值），caption 註明下限與未能排入的字數。")
A("")
A(f"圖檔共 **{n_figs}** 張，全部位於 `analysis/network-analysis/` 之下，`data/` 內無任何圖檔。"
  "圖中所有文字一律英文，唯一例外是教授姓名。")
A("")
A(f"另有 `analysis/network-analysis/by-figures/`：把 18 個單位的圖依**圖號**重新編排"
  f"（{by_fig_df['figure'].nunique()} 個目錄、{len(by_fig_df)} 個檔案，檔名為單位名稱），"
  "方便同一張圖跨單位並排比較；目錄清單見 `by_figures_index.csv`。"
  "此目錄每次執行都重建，與 `<unit>/figures/` 保持同步。")
A("")

A("## 限制與注意事項")
A("")
A("- 研究領域來自各校官網自填內容，深度與粒度本就不一致。網路密度差異可能反映資料填寫習慣，而非真實研究結構差異。")
A(f"- 覆蓋率差距達 {school_cmp['coverage_pct'].max() - school_cmp['coverage_pct'].min():.1f} 個百分點"
  f"（{school_cmp['coverage_pct'].min():.1f}% ~ {school_cmp['coverage_pct'].max():.1f}%）。覆蓋率不可解讀為系所或教授的品質排名。")
A("- `ntu-cs-ie` 與 `ntu-ds` 的學歷欄位為空，因此 PhD institution 相關屬性在這兩系所缺席。")
A("- Taxonomy 的 L1/L2 劃分為研究者判斷，不同劃分會影響 λ 項與社群對齊檢驗。"
  "`research_taxonomy.csv` 納入版本控管以確保可稽核與可重現。")
A("- λ 引入的推論連結是研究設計選擇，不是資料本身。所有主要結論須同時檢視 λ=0 的 baseline。")
A(f"- 有 {no_zh_total} 位進入網路的教授沒有中文姓名"
  f"（{'、'.join(f'`{k}` {v} 位' for k, v in no_zh_by_school.items())}），這些人在圖中顯示英文姓名。")
A("")
A("### 人工覆核裁定（去重規則 0）")
A("")
A("自動規則只看得到 email 與姓名字串：同一位教授在兩校使用不同網域的信箱時，規則 3 會保守地拆成兩個人。"
  f"這些案例已逐筆人工確認，裁定寫入 Stage 2 的 `MANUAL_SAME_PERSON` / `MANUAL_DIFFERENT_PERSON`"
  f"（共 {n_manual_same} 組確認為同一人、{n_manual_diff} 組確認為不同人），優先於所有自動規則：")
A("")
A(md_table(manual_decisions, ["decision", "label", "school_ids"],
           ["裁定", "教授", "school_id"]))
A("")
A(f"套用後，全體層級的跨校重複教授為 {n_cross_school_persons} 位、{n_cross_school_records} 筆紀錄。")
A("")
A("### 仍待人工覆核")
A("")
if len(collisions):
    A(f"其餘 {len(collisions)} 筆紀錄仍標記為 `name_collision_kept_separate`（同名、但 `name_en` 與 email 皆不相容），"
      "目前一律視為不同人：")
    A("")
    A(md_table(collisions.assign(emails=collisions["emails"].fillna("（無）")),
               ["professor_name", "name_en", "school_id", "emails"],
               ["professor_name", "name_en", "school_id", "email"]))
    A("")
    A("`資訊系` 兩筆是系所網頁把欄位標題當成姓名抓進來的雜訊紀錄，兩筆都沒有研究領域，不進入任何網路。"
      "`黃敬群` 兩筆的英文姓名完全不同（NCKU 為 `5180beb7123a6bf2@redacted.invalid`、研究 Operating Systems / Embedded Systems；"
      "NYCU 為 Ching-Chun Huang、研究 Machine Learning / Medical Imaging），依證據判定為兩位不同教授。"
      "若後續確認有誤，更新 Stage 2 的裁定表後重跑 Stage 2~4 即可。")
else:
    A("所有同名案例皆已裁定，無待覆核項目。")
A("")
A("### 被排除於身分比對的系所共用信箱")
A("")
A(f"下列 {len(shared_emails)} 個信箱在同一個 `school_id` 內重複出現，屬於系辦或聯絡窗口信箱，"
  "不是個人識別碼，已排除於 email 比對之外：")
A("")
A(md_table(shared_emails, ["email", "n_records", "school_ids"], ["email", "紀錄數", "school_id"], 0))
A("")

A("## 重現方式")
A("")
A("```powershell")
for nb, timeout in [("1_build_taxonomy", 1800), ("2_build_network_data", 1800),
                    ("3_network_analysis", 3600), ("4_group_and_cross_comparison", 1800)]:
    A(f"& 'python' -m jupyter nbconvert --to notebook "
      f"--execute --inplace --ExecutePreprocessor.timeout={timeout} "
      f"scripts/network-analysis/current-version/{nb}.ipynb")
A("```")
A("")
A("Stage 3 的虛無模型檢定（18 單位 × 1,000 次隨機化）為主要耗時來源，逾時設定較長。")
A("")

report = "\n".join(lines)
(DOCS / "network-analysis-report.md").write_text(report, encoding="utf-8")
print(f"written -> docs/network-analysis-report.md  ({len(report)} chars, {len(lines)} lines)")

written -> docs/network-analysis-report.md  (18463 chars, 260 lines)


## 驗證檢查清單

In [21]:
checks: list[tuple[str, bool, str]] = []

tax = pd.read_csv(NET_DIR / "research_taxonomy.csv", encoding="utf-8-sig")
checks.append((f"170 個 canonical 領域全數對應到 L2 與 L1，無孤兒",
               len(tax) == 170 and tax["parent_l2"].notna().all() and tax["parent_l1"].notna().all(), ""))
l2_per_l1 = tax.groupby("parent_l1")["parent_l2"].nunique()
overall_l1 = pd.read_csv(OUT_ROOT / "overall" / "l1_coverage_guardrail.csv", encoding="utf-8-sig")
checks.append(("每個 L1 至少包含 2 個 L2；無任一 L1 在全體覆蓋率超過 50%",
               bool((l2_per_l1 >= 2).all()) and bool((overall_l1["coverage_pct"] <= 50).all()),
               str(overall_l1[overall_l1["coverage_pct"] > 50].to_dict("records"))))

BASELINE = {"nccu-fin": 23, "nccu-mb": 18, "ncku-cs-ie": 51, "ncku-stats-ds": 39, "nthu-cs": 76,
            "nthu-qf": 17, "ntu-comm-engin-ds-network": 25, "ntu-cs-ie": 88, "ntu-ds": 59,
            "ntu-econ": 44, "ntu-fin": 32, "ntu-im": 30, "nycu-cs": 141, "nycu-info-mag-fin": 19}
bad = [s for s in SCHOOLS if METRICS[s]["bipartite"]["n_professors"] != BASELINE[s]]
checks.append(("各系所節點數與計畫 §1 覆蓋率表一致", not bad, str(bad)))

bad = [m["unit"] for m in MANIFEST if m["level"] in {"group", "overall"}
       and m["n_entities"] != m["n_records"] - m["n_duplicate_persons_removed"]]
checks.append(("組層級與全體層級去重後人數 = 原紀錄數 − 重複紀錄數", not bad, str(bad)))

all_kw = set(tax["keyword_en"])
bad = []
for m in MANIFEST:
    e = pd.read_csv(ROOT / m["out_dir"] / "prof_keyword_edges.csv", encoding="utf-8-sig")
    if set(e["keyword_en"]) - all_kw:
        bad.append(m["unit"])
checks.append(("邊表中所有 keyword 存在於 research_taxonomy.csv", not bad, str(bad)))

lam0 = lam_summary[lam_summary["lambda"] == 0.0]
checks.append((f"λ=0 baseline 已對 18 個單位輸出（{len(lam0)} 列）", len(lam0) == len(MANIFEST), ""))

need_cov = {"school_metrics_comparison.csv", "group_metrics_comparison.csv",
            "cross_school_bridging_summary.csv", "taxonomy_alignment.csv", "lambda_sensitivity_summary.csv"}
bad = [f for f in need_cov if "coverage_pct" not in
       pd.read_csv(CMP_DIR / f, encoding="utf-8-sig").columns]
checks.append(("所有比較表含 coverage 欄位", not bad, str(bad)))

bad = [m["unit"] for m in MANIFEST if "n_excluded_no_keywords" not in METRICS[m["unit"]]["coverage"]]
checks.append(("每個單位的 network_metrics.json 記錄排除人數", not bad, str(bad)))

bad = []
for m in MANIFEST:
    d = OUT_ROOT / m["unit"]
    for fname, key in [("prof_keyword_network.graphml", None),
                       ("keyword_network.graphml", "keyword_network"),
                       ("professor_network.graphml", "professor_network")]:
        g = nx.read_graphml(d / fname)
        expect = (METRICS[m["unit"]]["bipartite"]["n_professors"] + METRICS[m["unit"]]["bipartite"]["n_keywords"]
                  if key is None else METRICS[m["unit"]][key]["n_nodes"])
        if g.number_of_nodes() != expect:
            bad.append((m["unit"], fname))
checks.append(("GraphML 可被 networkx 重新讀入且節點數相符", not bad, str(bad[:5])))

checks.append((f"`person_identity_map.csv` 的 name_collision_kept_separate 已列於報告限制章節"
               f"（{len(collisions)} 筆）與人工覆核裁定（{len(manual_decisions)} 組）",
               "name_collision_kept_separate" in report and "人工覆核裁定" in report, ""))

stray = list((ROOT / "data").rglob("*.png")) + list((ROOT / "data").rglob("*.svg"))
checks.append(("所有 PNG 圖檔位於 analysis/network-analysis/ 之下，data/ 內無任何圖檔", not stray, str(stray[:5])))

counts = ALL_FIGS.groupby("unit").size().to_dict()
expected = {m["unit"]: (25 if m["level"] == "school" else 28) for m in MANIFEST}
expected["comparison"] = 12
bad = {u: (counts.get(u, 0), e) for u, e in expected.items() if counts.get(u, 0) != e}
checks.append((f"系所 25 張、組與全體 28 張、comparison 12 張，共 {sum(expected.values())} 張，無缺漏",
               not bad, str(bad)))

# 18~20（依學校上色、校 x 校矩陣、合聘標示）只有組層級與全體才有；
# 其餘圖號（含 21~28 的 word cloud）18 個單位都有——不是連續分段，不能用範圍推。
GROUP_ONLY_FIGS = {18, 19, 20}
n_units_all = len(MANIFEST)
n_units_group = sum(1 for m in MANIFEST if m["level"] != "school")
expected_by_fig = {
    i: (n_units_group if i in GROUP_ONLY_FIGS else n_units_all) for i in range(1, 29)
}
_actual = by_fig_df.groupby("figure").size().to_dict()
_bad_by_fig = {
    fig: (n, expected_by_fig[int(fig.split("_")[0])])
    for fig, n in _actual.items()
    if n != expected_by_fig[int(fig.split("_")[0])]
}
_orphans = [p for p in (OUT_ROOT / "by-figures").glob("*/*.png")
            if p.stem not in {m["unit"] for m in MANIFEST}]
checks.append((f"by-figures/ 依圖號重新編排完整（{by_fig_df['figure'].nunique()} 個目錄、"
               f"{len(by_fig_df)} 個檔案，無殘留）",
               not _bad_by_fig and not _orphans, f"{_bad_by_fig} {_orphans[:3]}"))

lang_all = pd.read_csv(OUT_ROOT / "figure_language_check.csv", encoding="utf-8-sig")
checks.append((f"語言檢查：{len(lang_all)} 張圖以程式掃描文字元素，除教授姓名外無中文字元",
               int(lang_all["n_violations"].sum()) == 0,
               str(lang_all[lang_all["n_violations"] > 0]["figure"].tolist()[:5])))

checks.append((f"無中文名的教授已記錄於 network_metrics.json（系所層級合計 {no_zh_total} 位）",
               no_zh_total > 0 and "沒有中文姓名" in report, ""))
checks.append((f"中文字型偵測成功（{CJK_FONT}）", FONT_OK, ""))

net_figs = [r for r in ALL_FIGS.itertuples(index=False)
            if r.figure.split("_")[0] in {"01", "02", "03", "04", "05", "06", "07", "08", "09", "18", "20"}
            and r.unit != "comparison"]
checks.append((f"所有網路圖 caption 皆註明節點數、邊數、backbone 與 λ（{len(net_figs)} 張）",
               len(net_figs) == 14 * 9 + 4 * 11, str(len(net_figs))))

for label, ok, detail in checks:
    print(f"[{'PASS' if ok else 'FAIL'}] {label}" + (f"  {detail}" if not ok else ""))

assert all(ok for _, ok, _ in checks), "Stage 4 驗證未通過"
print("\nStage 4 完成。")

[PASS] 170 個 canonical 領域全數對應到 L2 與 L1，無孤兒
[PASS] 每個 L1 至少包含 2 個 L2；無任一 L1 在全體覆蓋率超過 50%
[PASS] 各系所節點數與計畫 §1 覆蓋率表一致
[PASS] 組層級與全體層級去重後人數 = 原紀錄數 − 重複紀錄數
[PASS] 邊表中所有 keyword 存在於 research_taxonomy.csv
[PASS] λ=0 baseline 已對 18 個單位輸出（18 列）
[PASS] 所有比較表含 coverage 欄位
[PASS] 每個單位的 network_metrics.json 記錄排除人數
[PASS] GraphML 可被 networkx 重新讀入且節點數相符
[PASS] `person_identity_map.csv` 的 name_collision_kept_separate 已列於報告限制章節（4 筆）與人工覆核裁定（6 組）
[PASS] 所有 PNG 圖檔位於 analysis/network-analysis/ 之下，data/ 內無任何圖檔
[PASS] 系所 25 張、組與全體 28 張、comparison 12 張，共 474 張，無缺漏
[PASS] by-figures/ 依圖號重新編排完整（28 個目錄、462 個檔案，無殘留）
[PASS] 語言檢查：474 張圖以程式掃描文字元素，除教授姓名外無中文字元
[PASS] 無中文名的教授已記錄於 network_metrics.json（系所層級合計 13 位）
[PASS] 中文字型偵測成功（Microsoft JhengHei）
[PASS] 所有網路圖 caption 皆註明節點數、邊數、backbone 與 λ（170 張）

Stage 4 完成。
